# 01 — Vetores, Produto Interno e a Geometria dos Embeddings

**Módulo:** `00_deep_dive/math_linear_algebra`  
**Conexão com o curso:** Week 5 (RAG, cosine similarity) e Week 1 (embeddings de tokens)  
**Tempo estimado:** 2 horas  

---

## Por que isto importa

Na Week 5 você vai usar `cosine_similarity` para comparar embeddings em um pipeline RAG. A função vem pronta na biblioteca. Mas o que ela calcula, e por que esse número mede similaridade semântica?

Este notebook responde essa pergunta do zero, sem bibliotecas de ML, e mostra como o produto interno aparece diretamente na equação de atenção do Transformer:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

O produto $QK^\top$ é exatamente o cálculo do produto interno entre todos os pares de query e key.

## 1. Vetores: o que são no contexto de LLMs

Um embedding de token é um vetor $\mathbf{v} \in \mathbb{R}^d$, onde $d$ é a dimensão (tipicamente 768 a 4096 em modelos modernos). Cada dimensão representa uma característica latente aprendida durante o pré-treinamento.

Nós não sabemos o que cada dimensão significa individualmente — mas sabemos que vetores de palavras semanticamente relacionadas ficam próximos no espaço.

In [ ]:
# Python puro — zero dependências
import math
from typing import List

Vector = List[float]


def dot(a: Vector, b: Vector) -> float:
    """Produto interno: sum(a_i * b_i). Operação central do mecanismo de atenção."""
    if len(a) != len(b):
        raise ValueError(f"Dimensões incompatíveis: {len(a)} vs {len(b)}")
    return sum(ai * bi for ai, bi in zip(a, b))


def norm(v: Vector) -> float:
    """Norma L2 (euclidiana): sqrt(sum(v_i^2))."""
    return math.sqrt(dot(v, v))


def cosine_similarity(a: Vector, b: Vector) -> float:
    """
    cos(a, b) = <a, b> / (||a|| * ||b||)

    Resultado em [-1, 1]:
      +1 → mesma direção (semanticamente idênticos)
       0 → ortogonais (sem relação)
      -1 → direções opostas
    """
    denom = norm(a) * norm(b)
    if denom == 0:
        raise ValueError("Vetor zero não tem direção definida")
    return dot(a, b) / denom


print("Funções definidas.")

## 2. Experimento: geometria de embeddings simulados

Usamos vetores em 3D apenas para visualizar a intuição. Na prática, embeddings têm centenas ou milhares de dimensões.

In [ ]:
# Simulação de embeddings com 3 dimensões interpretáveis:
# [masculinidade, feminilidade, realeza]

embeddings = {
    "rei": [0.9, 0.1, 0.9],
    "rainha": [0.1, 0.9, 0.9],
    "homem": [0.9, 0.1, 0.1],
    "mulher": [0.1, 0.9, 0.1],
    "pedra": [0.0, 0.0, 0.0],  # nenhuma das características
}

palavras = list(embeddings.keys())

print(f"{'Par':<22} {'Cosine Similarity':>18}")
print("-" * 41)

pares = [
    ("rei", "rainha"),
    ("rei", "homem"),
    ("rainha", "mulher"),
    ("homem", "mulher"),
    ("rei", "pedra"),
]

for w1, w2 in pares:
    try:
        sim = cosine_similarity(embeddings[w1], embeddings[w2])
        print(f"{w1} × {w2:<17} {sim:>18.4f}")
    except ValueError:
        print(f"{w1} × {w2:<17} {'indefinida (vetor zero)':>18}")

**Observe:** rei e rainha têm similaridade alta (mesma categoria de realeza); rei e pedra têm similaridade indefinida porque pedra é o vetor zero.

Na prática, embeddings reais não têm interpretação por dimensão — mas a geometria é a mesma: palavras semanticamente próximas têm cosine similarity alta.

## 3. Analogias vetoriais: rei - homem + mulher ≈ rainha

A propriedade mais famosa de word embeddings: relações semânticas são relações vetoriais.

In [ ]:
def subtract(a: Vector, b: Vector) -> Vector:
    return [ai - bi for ai, bi in zip(a, b)]


def add(a: Vector, b: Vector) -> Vector:
    return [ai + bi for ai, bi in zip(a, b)]


# rei - homem + mulher
resultado = add(subtract(embeddings["rei"], embeddings["homem"]), embeddings["mulher"])
print(f"rei - homem + mulher = {resultado}")
print(f"rainha               = {embeddings['rainha']}")
print()

# Qual palavra do vocabulário é mais próxima do resultado?
print("Similaridade do resultado com cada palavra:")
for palavra, emb in embeddings.items():
    try:
        sim = cosine_similarity(resultado, emb)
        print(f"  {palavra:<10} {sim:.4f}")
    except ValueError:
        print(f"  {palavra:<10} indefinida")

## 4. Conexão direta com o mecanismo de atenção

O produto interno aparece em $QK^\top$: para cada token $i$ (query), calculamos o produto interno com todos os outros tokens $j$ (keys). O resultado indica o quanto o token $i$ deve "prestar atenção" ao token $j$.

In [ ]:
def softmax(scores: Vector) -> Vector:
    """Softmax numericamente estável."""
    m = max(scores)
    exps = [math.exp(s - m) for s in scores]
    total = sum(exps)
    return [e / total for e in exps]


def scaled_dot_product_attention(
    query: Vector,
    keys: List[Vector],
    values: List[Vector],
) -> Vector:
    """
    Atenção escalar (single-head, single query).

    Eq. do paper 'Attention Is All You Need' (Vaswani et al., 2017)
    aplicada a um único vetor query e uma sequência de keys/values.
    """
    d_k = len(query)
    # Scores: produto interno query × cada key, escalado por sqrt(d_k)
    scores = [dot(query, k) / math.sqrt(d_k) for k in keys]
    # Pesos de atenção via softmax
    weights = softmax(scores)
    # Saída: soma ponderada dos values
    d_v = len(values[0])
    output = [0.0] * d_v
    for w, v in zip(weights, values):
        for i in range(d_v):
            output[i] += w * v[i]
    return output, weights


# Exemplo: query alinhada com key[0]
q = [1.0, 0.0, 0.0]
keys = [[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]
values = [[10.0], [20.0], [30.0]]

output, weights = scaled_dot_product_attention(q, keys, values)
print(f"Pesos de atenção: {[f'{w:.4f}' for w in weights]}")
print(f"Saída (deve dominar value[0]=10): {output}")

## 5. Verificação com NumPy

Confirma que nossa implementação manual bate com a vetorizada.

In [ ]:
import numpy as np

q_np = np.array([1.0, 0.0, 0.0])
K_np = np.array([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]])
V_np = np.array([[10.0], [20.0], [30.0]])

scores_np = (q_np @ K_np.T) / np.sqrt(len(q_np))
weights_np = np.exp(scores_np - scores_np.max())
weights_np /= weights_np.sum()
output_np = weights_np @ V_np

print(f"NumPy:      {output_np.flatten()}")
print(f"Manual:     {output}")

diff = abs(output_np[0, 0] - output[0])
print(f"Diferença:  {diff:.2e}")
assert diff < 1e-10, "Divergência detectada!"
print("OK — implementações idênticas.")

## Resumo

- **Produto interno** mede a projeção de um vetor sobre outro — a similaridade direcional.
- **Cosine similarity** normaliza isso para $[-1, 1]$, independente da magnitude.
- **Scaled dot-product attention** usa produto interno para calcular relevância entre tokens.

Na Week 5 você vai chamar `cosine_similarity` de uma biblioteca. Agora você sabe o que ela calcula geometricamente e por que funciona para medir similaridade semântica.

**Próximo:** `02_matrix_multiplication.ipynb` — como a multiplicação matricial permite calcular atenção para toda a sequência simultaneamente.